In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score, roc_curve, f1_score,
                             mean_squared_error, mean_absolute_error, r2_score)
from sklearn.metrics import confusion_matrix, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/creditcard.csv', encoding='latin-1')

In [ ]:
print(df.columns)
df.info()
df.shape

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49610 entries, 0 to 49609
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    49610 non-null  int64  
 1   V1      49610 non-null  float64
 2   V2      49610 non-null  float64
 3   V3      49610 non-null  float64
 4   V4      49609 non-null  float64
 5   V5      49609 non-null  float64
 6   V6      49609 non-null  float64
 7   V7      49609 non-null  float64
 8   V8      49609 non-null  float64
 9   V9      49609 non-null  float64
 10  V10     49609 non-null  float64
 11  V11     49609 non-null  float64
 12  V12     49609 non-null  float64
 13  V13     49609 non-null  float64
 14  V14     49609 non-null  float

(49610, 31)

In [ ]:
df.describe().round(3)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,49610.000,49610.000,49610.000,49610.000,49609.000,49609.000,49609.000,49609.000,49609.000,49609.000,...,49609.000,49609.000,49609.000,49609.000,49609.000,49609.000,49609.000,49609.000,49609.000,49609.000
mean,28803.556,-0.243,0.012,0.693,0.185,-0.257,0.104,-0.120,0.053,0.123,...,-0.028,-0.107,-0.040,0.008,0.136,0.021,0.005,0.005,93.121,0.003
std,13097.469,1.886,1.631,1.511,1.400,1.413,1.311,1.284,1.224,1.213,...,0.736,0.638,0.591,0.594,0.439,0.501,0.388,0.333,253.266,0.055
min,0.000,-56.408,-72.716,-32.965,-5.173,-42.148,-26.161,-26.548,-41.485,-9.284,...,-20.262,-8.594,-26.751,-2.837,-7.496,-1.577,-8.568,-9.618,0.000,0.000
25%,21734.250,-0.993,-0.563,0.218,-0.721,-0.866,-0.636,-0.606,-0.147,-0.611,...,-0.232,-0.530,-0.179,-0.322,-0.128,-0.331,-0.063,-0.007,7.610,0.000
50%,33390.000,-0.247,0.079,0.797,0.190,-0.288,-0.151,-0.077,0.058,0.012,...,-0.068,-0.082,-0.052,0.062,0.176,-0.072,0.009,0.022,25.000,0.000
75%,38852.750,1.156,0.732,1.431,1.067,0.284,0.494,0.425,0.332,0.819,...,0.108,0.307,0.078,0.401,0.422,0.300,0.084,0.076,85.000,0.000
max,44135.000,1.960,18.184,4.102,16.491,34.802,22.529,36.677,20.007,10.393,...,22.615,5.806,17.298,4.014,5.525,3.517,11.136,33.848,12910.930,1.000


In [ ]:
df.head(10).round(3)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.360,-0.073,2.536,1.378,-0.338,0.462,0.240,0.099,0.364,...,-0.018,0.278,-0.110,0.067,0.129,-0.189,0.134,-0.021,149.62,0.0
1,0,1.192,0.266,0.166,0.448,0.060,-0.082,-0.079,0.085,-0.255,...,-0.226,-0.639,0.101,-0.340,0.167,0.126,-0.009,0.015,2.69,0.0
2,1,-1.358,-1.340,1.773,0.380,-0.503,1.800,0.791,0.248,-1.515,...,0.248,0.772,0.909,-0.689,-0.328,-0.139,-0.055,-0.060,378.66,0.0
3,1,-0.966,-0.185,1.793,-0.863,-0.010,1.247,0.238,0.377,-1.387,...,-0.108,0.005,-0.190,-1.176,0.647,-0.222,0.063,0.061,123.50,0.0
4,2,-1.158,0.878,1.549,0.403,-0.407,0.096,0.593,-0.271,0.818,...,-0.009,0.798,-0.137,0.141,-0.206,0.502,0.219,0.215,69.99,0.0
5,2,-0.426,0.961,1.141,-0.168,0.421,-0.030,0.476,0.260,-0.569,...,-0.208,-0.560,-0.026,-0.371,-0.233,0.106,0.254,0.081,3.67,0.0
6,4,1.230,0.141,0.045,1.203,0.192,0.273,-0.005,0.081,0.465,...,-0.168,-0.271,-0.154,-0.780,0.750,-0.257,0.035,0.005,4.99,0.0
7,7,-0.644,1.418,1.074,-0.492,0.949,0.428,1.121,-3.808,0.615,...,1.943,-1.015,0.058,-0.650,-0.415,-0.052,-1.207,-1.085,40.80,0.0
8,7,-0.894,0.286,-0.113,-0.272,2.670,3.722,0.370,0.851,-0.392,...,-0.073,-0.268,-0.204,1.012,0.373,-0.384,0.012,0.142,93.20,0.0
9,9,-0.338,1.120,1.044,-0.222,0.499,-0.247,0.652,0.070,-0.737,...,-0.247,-0.634,-0.121,-0.385,-0.070,0.094,0.246,0.083,3.68,0.0


In [ ]:
df.isna().sum()

,0
Time,0
V1,0
V2,0
V3,0
V4,1
V5,1
V6,1
V7,1
V8,1
V9,1


In [ ]:
df = df.dropna(subset=['Class'])

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
num_features = ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
                'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22',
                 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

df = df.dropna(subset=num_features)

pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', StandardScaler(), num_features),

    ])),
    ('classifier', xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss', verbosity=0))
])

pipeline.fit(X_train, y_train)
y_prob = pipeline.predict_proba(X_test)[:, 1]

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

best_thr = 0.69
y_pred = (y_prob >= best_thr).astype(int)
accuracy = accuracy_score(y_test, y_pred)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy*100:.1f}%")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob)*100:.1f}%")
print(f"F1-Score:  {f1_score(y_test, y_pred)*100:.1f}%")

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

Accuracy: 100.0%
ROC-AUC:   97.5%
F1-Score:  93.1%
[[9891    1]
 [   3   27]]
              precision    recall  f1-score   support

         0.0     0.9997    0.9999    0.9998      9892
         1.0     0.9643    0.9000    0.9310        30

    accuracy                         0.9996      9922
   macro avg     0.9820    0.9499    0.9654      9922
weighted avg     0.9996    0.9996    0.9996      9922



In [ ]:
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

print()
print("0/1 count (counts):")
print(y.value_counts())
print("\n\0/1 on percentages(%):")
print((y.value_counts(normalize=True)*100).round(2))

Train: 39687, Test: 9922

0/1 count (counts):
Class
0.0    49461
1.0      148
Name: count, dtype: int64

 /1 on percentages(%):
Class
0.0    99.7
1.0     0.3
Name: proportion, dtype: float64


In [ ]:
thresholds = np.linspace(0.01, 1, 50)
best_thr = 0.5
best_f1 = 0

for thr in thresholds:
    y_pred_thr = (y_prob >= thr).astype(int)
    f1 = f1_score(y_test, y_pred_thr)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

print("Best threshold:", best_thr)
print("Best F1:", best_f1)

Best threshold: 0.6969387755102041
Best F1: 0.9310344827586207


# Saving model

In [ ]:
import joblib, os

joblib.dump(pipeline, 'fraud_detection.joblib')
print(f"✅ Сохранено: credit_scoring_model.joblib ({os.path.getsize('fraud_detection.joblib')/1024:.0f} KB)")

loaded = joblib.load('fraud_detection.joblib')

✅ Сохранено: credit_scoring_model.joblib (198 KB)


In [ ]:
%%writefile app.py

from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)
app.json.ensure_ascii = False

model = joblib.load('fraud_detection.joblib')


@app.route('/')
def home():
    return jsonify({
        'service': 'Fraud Detection API',
        'status': 'working',
        'usage': 'Send POST request to /predict'
    })


@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()

        if not data:
            return jsonify({'error': 'Body is Empty , send JSON!!.'}), 400

        required_fields = [
            'V1', 'V2', 'V3', 'V4',
            'V5', 'V6',
            'V7', 'V8', 'V9','V10','V11','V12',
            'V13','V14','V15','V16','V17','V18','V19',
            'V20','V21','V22','V23','V24','V25','V26',
            'V27','V28','Amount'
        ]
        missing = [f for f in required_fields if f not in data]
        if missing:
            return jsonify({
                'error': f'send this columns also: {missing}',
                'required': required_fields
            }), 400

        client_df = pd.DataFrame([data])

        prediction = model.predict(client_df)[0]
        probability = model.predict_proba(client_df)[0].tolist()

        result = {
            'prediction': int(prediction),
            'result': 'Fraud' if prediction == 1 else 'not Fraud',
            'probability_reject': round(probability[0], 4),
            'probability_approve': round(probability[1], 4)
        }

        return jsonify(result)

    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/info')
def info():
    return jsonify({
        'model': 'XGBoost',
        'features': ['V1', 'V2', 'V3', 'V4',
            'V5', 'V6',
            'V7', 'V8', 'V9','V10','V11','V12',
            'V13','V14','V15','V16','V17','V18','V19',
            'V20','V21','V22','V23','V24','V25','V26',
            'V27','V28','Amount'],
        'target': 'Class (0=not Fraud, 1=Fraud)'
    })


if __name__ == '__main__':
    print()
    print("=" * 50)
    print("  Server is running!")
    print("  http://localhost:5000")
    print("=" * 50)
    print()
    app.run(debug=False, host='0.0.0.0', port=5000)

Writing app.py


In [ ]:
from pyngrok import ngrok

# Если у вас есть токен ngrok — вставьте его:
ngrok.set_auth_token("3ATNNTRsOvyD3pwRqvNcdn77kM3_6aVsgnM9fuNBaigjBKRZ2")

# Создаём туннель
public_url = ngrok.connect(5000)

print("=" * 60)
print("  🌐 Сервер доступен из интернета!")
print(f"  📌 URL: {public_url}")
print("=" * 60)
print()
print("👉 Откройте этот URL в браузере (новая вкладка)!")
print("   Вы увидите: 'Привет! Сервер работает 🚀'")
print()
print(f"👉 Попробуйте: {public_url}/about")
print("   Вы увидите: 'Это учебный ML-сервер.'")
print()

BASE_URL = public_url.public_url

  🌐 Сервер доступен из интернета!
  📌 URL: NgrokTunnel: "https://uninstructive-nonfraudulently-xenia.ngrok-free.dev" -> "http://localhost:5000"

👉 Откройте этот URL в браузере (новая вкладка)!
   Вы увидите: 'Привет! Сервер работает 🚀'

👉 Попробуйте: NgrokTunnel: "https://uninstructive-nonfraudulently-xenia.ngrok-free.dev" -> "http://localhost:5000"/about
   Вы увидите: 'Это учебный ML-сервер.'



In [ ]:
import requests

data = {
    'V1': -1.2,
    'V2': 0.3,
    'V3': 1.1,
    'V4': -0.5,
    'V5': 0.2,
    'V6': -0.1,
    'V7': 0.5,
    'V8': -0.2,
    'V9': 0.1,
    'V10': -0.3,
    'V11': 0.4,
    'V12': -0.1,
    'V13': 0.2,
    'V14': -0.5,
    'V15': 0.1,
    'V16': -0.2,
    'V17': 0.3,
    'V18': -0.4,
    'V19': 0.2,
    'V20': 0.1,
    'V21': -0.3,
    'V22': 0.2,
    'V23': -0.1,
    'V24': 0.4,
    'V25': -0.2,
    'V26': 0.1,
    'V27': -0.1,
    'V28': 0.05,
    'Amount': 1500
}

r = requests.post(f"{BASE_URL}/predict", json=data)

res = r.json()

print("Result:", res['result'])
print("Fraud probability:", res['probability_approve'])

Result: not Fraud
Fraud probability: 0.0
